In [28]:
import numpy as np
from numpy.linalg import inv as inv

In [29]:
def ten2mat(tensor, mode):
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order = 'F')

def mat2ten(mat, tensor_size, mode):
    index = list()
    index.append(mode)
    for i in range(tensor_size.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(tensor_size[index]), order = 'F'), 0, mode)

def svt_tnn(mat, alpha, rho, theta):
    """This is a Numpy dependent singular value thresholding (SVT) process."""
    u, s, v = np.linalg.svd(mat, full_matrices = 0)
    vec = s.copy()
    vec[theta :] = s[theta :] - alpha / rho
    vec[vec < 0] = 0
    return u @ np.diag(vec) @ v

def svt_tnn(mat, alpha, rho, theta):
    tau = alpha / rho
    [m, n] = mat.shape
    if 2 * m < n:
        u, s, v = np.linalg.svd(mat @ mat.T, full_matrices = 0)
        s = np.sqrt(s)
        idx = np.sum(s > tau)
        mid = np.zeros(idx)
        mid[:theta] = 1
        mid[theta:idx] = (s[theta:idx] - tau) / s[theta:idx]
        return (u[:, :idx] @ np.diag(mid)) @ (u[:, :idx].T @ mat)
    elif m > 2 * n:
        return svt_tnn(mat.T, alpha, rho, theta).T
    u, s, v = np.linalg.svd(mat, full_matrices = 0)
    idx = np.sum(s > tau)
    vec = s[:idx].copy()
    vec[theta:idx] = s[theta:idx] - tau
    return u[:, :idx] @ np.diag(vec) @ v[:idx, :]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])
    
def compute_mape(var, var_hat):
    return np.sum(np.abs(var - var_hat) / var) / var.shape[0]

def LRTC(dense_tensor, sparse_tensor, alpha, rho, theta, epsilon, maxiter):
    """Low-Rank Tenor Completion with Truncated Nuclear Norm, LRTC-TNN."""
    
    dim = np.array(sparse_tensor.shape)
    pos_missing = np.where(sparse_tensor == 0)
    pos_test = np.where((dense_tensor != 0) & (sparse_tensor == 0))
    
    X = np.zeros(np.insert(dim, 0, len(dim))) # \boldsymbol{\mathcal{X}}
    T = np.zeros(np.insert(dim, 0, len(dim))) # \boldsymbol{\mathcal{T}}
    Z = sparse_tensor.copy()
    last_tensor = sparse_tensor.copy()
    snorm = np.sqrt(np.sum(sparse_tensor ** 2))
    it = 0
    while True:
        rho = min(rho * 1.05, 1e5)
        for k in range(len(dim)):
            X[k] = mat2ten(svt_tnn(ten2mat(Z - T[k] / rho, k), alpha[k], rho, int(np.ceil(theta * dim[k]))), dim, k)
        Z[pos_missing] = np.mean(X + T / rho, axis = 0)[pos_missing]
        T = T + rho * (X - np.broadcast_to(Z, np.insert(dim, 0, len(dim))))
        tensor_hat = np.einsum('k, kmnt -> mnt', alpha, X)
        tol = np.sqrt(np.sum((tensor_hat - last_tensor) ** 2)) / snorm
        last_tensor = tensor_hat.copy()
        it += 1
        if (it + 1) % 50 == 0:
            print('Iter: {}'.format(it + 1))
            print('RMSE: {:.6}'.format(compute_rmse(dense_tensor[pos_test], tensor_hat[pos_test])))
            print()
        if (tol < epsilon) or (it >= maxiter):
            break

    print('Imputation MAPE: {:.6}'.format(compute_mape(dense_tensor[pos_test], tensor_hat[pos_test])))
    print('Imputation RMSE: {:.6}'.format(compute_rmse(dense_tensor[pos_test], tensor_hat[pos_test])))
    print()
    
    return tensor_hat

    

In [30]:
import numpy as np
import scipy.io
import pandas as pd
import time
from pathlib import Path

# ---------- locate project root ----------
def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "Main data").exists():
            return parent
    raise RuntimeError("Project root (CODE UPLOAD) not found.")

PROJECT_ROOT = find_project_root()

# ==== Step 1: 读取数据 ====
mat_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "local_speed_data.mat"
)

data = scipy.io.loadmat(str(mat_path))
sparse_tensor = data['speed_data'].astype(float)

# 标记 >40 的异常值为缺失 NaN
sparse_tensor[sparse_tensor > 40] = np.nan

# 保存一份原始观测值
dense_tensor = sparse_tensor.copy()

# 用 0 替代 NaN，供算法内部使用（缺失为0）
sparse_tensor = np.nan_to_num(sparse_tensor, nan=0.0)

# ==== Step 2: 设置补全参数 ====
alpha = np.ones(3) / 3
rho = 1e-5
theta = 0.2
epsilon = 1e-4
maxiter = 100

# ==== Step 3: 执行补全 ====
start = time.time()
tensor_hat = LRTC(dense_tensor, sparse_tensor, alpha, rho, theta, epsilon, maxiter)
end = time.time()
print(f"补全过程耗时: {end - start:.2f} 秒")




Iter: 50
RMSE: nan

Iter: 100
RMSE: nan

Imputation MAPE: nan
Imputation RMSE: nan

补全过程耗时: 1875.72 秒


In [31]:
# ==== Step 4: 还原非缺失值 ====
mask_observed = ~np.isnan(dense_tensor)
tensor_hat[mask_observed] = dense_tensor[mask_observed]
# ==== Step 5: 将补全失败的缺失值设回 NaN ====
missing_mask = np.isnan(dense_tensor)
tensor_hat[missing_mask & (tensor_hat == 0)] = np.nan

In [32]:
# ==== Step 6: 展开为 2D 表格 [day*batch, grid] 并保存 ====
# 将负值设置为 0（非常重要）
tensor_hat[tensor_hat < 0] = 0

# 展开张量 [grid, day, batch] -> [day * batch, grid]
grid, day, batch = tensor_hat.shape
tensor_2d = tensor_hat.transpose(1, 2, 0).reshape(day * batch, grid)

In [33]:
# === 加载参考 CSV 的行列名（relative）===
csv_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "merged_local_avg_speed.csv"
)
merged_df = pd.read_csv(csv_path, index_col=0)
index_names = merged_df.index
column_names = merged_df.columns

# === 自动对齐行列名 ===
if tensor_2d.shape[0] != len(index_names):
    index_names = [f"row_{i}" for i in range(tensor_2d.shape[0])]

if tensor_2d.shape[1] != len(column_names):
    column_names = [f"grid_{i}" for i in range(tensor_2d.shape[1])]

# === 保存为 CSV（relative）===
output_df = pd.DataFrame(tensor_2d, index=index_names, columns=column_names)

output_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "completed_tensor.csv"
)
output_df.to_csv(output_path)

print(f"✅ 补全结果已保存为 CSV：{output_path}")

✅ 补全结果已保存为 CSV：C:\DCLASS\myJupyter\paper2\shanghai_data_save\local_grid_time-series\completed_tensor.csv


In [ ]:
adj_columns = df_adj.columns.tolist()  # 是整数列表

# 读取 completed_tensor.csv（relative）
csv_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "completed_tensor.csv"
)

df = pd.read_csv(csv_path)

# 将列名转换为整数匹配（除第一列 'batch'）
original_columns = df.columns.tolist()
first_column = original_columns[0]  # 通常是 'batch'

# 保留第一列 + adj 中存在的列（注意列名类型转换）
columns_to_keep = [first_column] + [str(c) for c in adj_columns if str(c) in original_columns]

# 筛选数据
filtered_df = df[columns_to_keep]
filtered_df

In [ ]:
# ---------- OUTPUT 1: completed_tensor_filtered.csv (relative) ----------
out_dir_1 = PROJECT_ROOT / "shanghai_data_save" / "local_grid_time-series"
out_dir_1.mkdir(parents=True, exist_ok=True)

out_csv_1 = out_dir_1 / "completed_tensor_filtered.csv"
filtered_df.to_csv(out_csv_1, index=False, encoding="utf-8-sig")

print("✅ 已筛选并保存与邻接矩阵列名一致的 completed_tensor.csv")

# ---------- OUTPUT 2: feature_matrix_X.csv (relative) ----------
out_dir_2 = PROJECT_ROOT / "Main data" / "prediction_data"
out_dir_2.mkdir(parents=True, exist_ok=True)

out_csv_2 = out_dir_2 / "feature_matrix_X.csv"
filtered_df.to_csv(out_csv_2, index=False, header=False, encoding="utf-8-sig")

print("✅ 已保存为纯数值格式 feature_matrix_X.csv（prediction_data）")